In [1]:
import boto3
import botocore
import os
from botocore.exceptions import ClientError

In [2]:
session = boto3.Session(profile_name='default')

In [8]:
S3_CLIENT = session.client('s3')

In [9]:
BUCKET_NAME = 'codespaces-bucket-sample-1983'

## Automatizador de subida de archivos a Amazon S3

**Objetivo**

Crear un script en Python que suba automáticamente archivos locales a un bucket.

**Servicios involucrados**
- Amazon S3
- boto3

**Funcionalidad del proyecto**
El script debe:
- Subir un archivo
- Listar archivos en el bucket
- Descargar un archivo

In [11]:
# Automatizador de subida de archivos a Amazon S3

def existe_archivo(archivo):
    return os.path.exists(archivo)

def listar_objetos_de_bucket(bucket):
    try:
        response = S3_CLIENT.list_objects_v2(
            Bucket=bucket
        )
    except ClientError as error:
        print(f"Ocurrio un error : {error}")
        return None
    
    return response
    


def listar_archivos_de_bucket(bucket):
    response = listar_objetos_de_bucket(bucket)
    
    if response is None:
        return

    contents = response['Contents']
    name = response['Name']

    print(f"Contenido de Bucket : {name}")
    print(f"----------------------------")

    for item in contents:
        key = item['Key']
        lastModified = item['LastModified']

        print(f"OBJETO : {key}  -> Ultima modificacion: {lastModified}")

def subir_archivo_a_bucket(archivo, bucket, key=None):
    
    if not existe_archivo(archivo):
        print("El archivo no existe! Intenta de nuevo")
        return
    
    if key is None:
        key  = archivo

    try:
        response = S3_CLIENT.put_object(
            Body=archivo,
            Bucket=bucket,
            Key=key
        )

        eTag = response['ETag']

        print("Se ha subido el archivo exitosamente")
        print(f"ETag : {eTag}\n")

    except ClientError as error:
        print(f"Ocurrio un error : {error}")
        return


In [ ]:
archivo_prueba='archivo_prueba.txt'

print(f"Subiendo archivo : {archivo_prueba}")

subir_archivo_a_bucket(
    archivo=archivo_prueba, 
    bucket=BUCKET_NAME
)

print(f"Listando objetos en bucket : {BUCKET_NAME}")

listar_archivos_de_bucket(bucket=BUCKET_NAME)

Subiendo archivo : archivo_prueba.txt
Se ha subido el archivo exitosamente
ETag : "872c76819acc3fb9b0922bc1b4d34284"

Listando objetos en bucket : codespaces-bucket-sample-1983
Contenido de Bucket : codespaces-bucket-sample-1983
----------------------------
OBJETO : archivo_prueba.txt  -> Ultima modificacion: 2026-04-10 02:54:54+00:00


## Sistema automático de Backups a S3

Crear un script que **haga backup automático de una carpeta local** hacia S3.

**Servicios involucrados**
- Amazon S3

**Funcionalidad**
El programa debe:

- Escanear una carpeta
- Subir todos los archivos
- Crear una carpeta con fecha

In [12]:
from datetime import datetime

def recuperar_elementos_recursivos_de_carpeta(carpeta):
        root_path, dirnames, filenames = list(os.walk("carpeta-prueba"))[0]
        return root_path, dirnames, filenames

def elemento_es_directorio(carpeta):
        return os.path.isdir(carpeta)

def generar_nombre_de_carpeta_unico(carpeta):
        return carpeta + "-" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

def respaldar_carpeta_a_bucket_S3(carpeta, bucket):
        
        if not elemento_es_directorio(carpeta):
                print("La carpeta introducida no es valida! Intenta de nuevo")
                return
    
        root_path, _, filenames = recuperar_elementos_recursivos_de_carpeta(carpeta)

        unique_root_path = generar_nombre_de_carpeta_unico(root_path)

        for filename in filenames:      
                
                print(f"Subiendo archivo : {filename}")

                subir_archivo_a_bucket(
                archivo=os.path.join(root_path, filename), 
                bucket=bucket,
                key=os.path.join(unique_root_path, filename)
                )

In [10]:
respaldar_carpeta_a_bucket_S3("carpeta-prueba", "bucket-test-lmsf-1982")

Subiendo archivo : texto3.txt
Se ha subido el archivo exitosamente
ETag : "ade747e14441cd5c4340f7b0f14665e4"

Subiendo archivo : texto2.txt
Se ha subido el archivo exitosamente
ETag : "4353095f192338f3a00c5cf952bf02ed"

Subiendo archivo : texto1.txt
Se ha subido el archivo exitosamente
ETag : "080597d5efbaf458ee6113f69dfbfb56"



## 3. Monitor de instancias EC2
Objetivo: Crear un script que liste y monitoree instancias EC2.
Servicios involucrados

- Amazon EC2

**Funcionalidad**

El script debe mostrar:
- Instance ID
- Estado
- Tipo de instancia
- IP pública

In [11]:
EC2_CLIENT = session.client('ec2')

In [22]:

def listar_instancias():
    try:
        response = EC2_CLIENT.describe_instances()
    except ClientError as err:
        print(f"Ocurrio un error! : {err}")

    reservations = response['Reservations']

    if len(reservations) == 0:
        print("No hay instancias en la region. Intenta crear una")
        return
    
    instances = reservations[0]["Instances"]

    for instance in instances:
        print(f"INSTANCIA ID: {instance['InstanceId']}")
        print(f"Estado: {instance['State']}")
        print(f"Tipo de Instancia: {instance['InstanceType']}")

        print(f"IP Pública: {instance.get('PublicIpAddress', "-")}")
        print("\n-------------------------------------\n")





In [23]:
listar_instancias()

INSTANCIA ID: i-0bb2a6c564c1af73a
Estado: {'Code': 80, 'Name': 'stopped'}
Tipo de Instancia: t3.micro
IP Pública: -

-------------------------------------

INSTANCIA ID: i-0486d38fbf218ac40
Estado: {'Code': 48, 'Name': 'terminated'}
Tipo de Instancia: t3.micro
IP Pública: -

-------------------------------------



## 4. Script para limpiar buckets S3 automáticamente
**Objetivo**
Crear un programa que detecte y elimine archivos antiguos.

**Servicios**
- Amazon S3

**Funcionalidad**

El script debe:
- listar objetos
- detectar archivos > 30 días
- eliminarlos

In [26]:
from datetime import datetime, timedelta, timezone

def obtener_fecha_de_mes_pasado():
    hoy = datetime.now(timezone.utc)
    return  hoy - timedelta(days=30)


def limpieza_de_objetos_antiguos_en_bucket_S3(bucket):
    num_objetos_eliminados = 0

    response = listar_objetos_de_bucket(bucket=BUCKET_NAME)
    contents = response['Contents']

    umbral = obtener_fecha_de_mes_pasado()

    print(f"OBJETOS DEL BUCKET : {bucket}")
    
    for item in response['Contents']:
        key = item['Key']
        lastModified = item['LastModified']

        print(f"Objeto : {key}\t\t\tFecha de ultima modificación: { lastModified.strftime("%d-%m-%Y")}")

        if lastModified < umbral:
            print("Objeto de más de 30 días. Eliminando.....")

            try:
                S3_CLIENT.delete_object(
                    Bucket=bucket,
                    Key=key
                )
            except ClientError as err:
                print(f"Hubo un error! : {err}")
            else:
                num_objetos_eliminados += 1

    print(f"Se han eliminado {num_objetos_eliminados} objetos antiguos (>30 días)")


In [27]:
limpieza_de_objetos_antiguos_en_bucket_S3(BUCKET_NAME)

OBJETOS DEL BUCKET : codespaces-bucket-sample-1983
Objeto : archivo_prueba.txt			Fecha de ultima modificación: 10-04-2026
Se han eliminado 0 objetos antiguos (>30 días)


## 7. Inventario de recursos AWS
**Objetivo**
- Crear un script que escanee tu cuenta AWS y genere un inventario.

**Servicios:**
- Amazon EC2
- Amazon S3
- AWS Lambda

**Salida del programa**
El script genera un archivo:
- aws_inventory.json

In [3]:
RE_CLIENT = session.client('resource-explorer-2')

In [16]:
import json

def guardar_a_archivo_JSON(inventario):
    with open("aws_inventory.json", "w") as inventory_file:
        json.dump(inventario, inventory_file, indent=4)

    print("Guardado inventario de recursos AWS a archivo 'aws_inventory'!")

def guardar_inventario_de_recursos_AWS():
    RE_CLIENT = session.client('resource-explorer-2')

    try:
        recursos = RE_CLIENT.list_resources(
            Filters={
                    'FilterString': 'resourcetype:s3:bucket,ec2:instance,lambda:function'
            },
            MaxResults=100
        )['Resources']

    except ClientError as err:
        print(f"Hubo un error! : {err}")
        
    else:
        inventario = []

        for recurso in recursos:
            item = {}
            item['Arn'] = recurso['Arn']
            item['ResourceType'] = recurso['ResourceType']
            item['Service'] = recurso['Service']

            inventario.append(item)

        guardar_a_archivo_JSON(inventario)
        

In [17]:
guardar_inventario_de_recursos_AWS()

Guardado inventario de recursos AWS a archivo 'aws_inventory'!
